# 人在迴路中：前置行動閘門、風險分層與審計記錄

本課程的 README 介紹了人在迴路中（Human-in-the-Loop）的簡短範例，該範例在代理已經產生回應後，要求使用者進行「批准」或「拒絕」。這種模式是個不錯的起點，但實際生產環境中的 HITL 實作通常需要額外三個要素：

1. 一個<strong>前置行動閘門</strong>，在代理執行風險步驟<strong>之前</strong>運作，以保持成本、不可逆性和延遲在可控範圍內。
2. <strong>風險分層</strong>，使低風險行動自動執行、中風險行動批次批准，唯有高風險行動才需人工阻擋。
3. 一個<strong>審計日誌與修改迴圈</strong>，記錄每次閘門決策為 JSONL 格式，拒絕時用結構化原因重新提示代理，而非僅顯示「正在修改...」。

這個筆記本基於與 `06-system-message-framework.ipynb` 相同的基礎構建每項功能。它可在 `DEMO_MODE = True`（不需互動輸入）或 `DEMO_MODE = False` 時使用真實的 `input()` 提示下端到端執行。注意：在 DEMO_MODE 下，第三目標的重試是腳本化的，以便完整展示迴圈機制。真正依靠修改驅動的重新分類需要 `DEMO_MODE = False` 且由操作員執行。

**不在範圍內（由其他課程處理）：** 認證及存取控制（課程 06 README 威脅 2）、工具呼叫中介軟體（課程 14 MAF 深入探討）、多代理辯論模式。


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## 範例 1：預先行動閘口

README 中的 HITL 程式碼片段是先呼叫代理人，然後再請使用者批准輸出。那是一個<strong>事後行動</strong>流程。代理人已經執行過，所以 LLM 呼叫費用已經支付，任何副作用（寄出的郵件、寫入的資料庫行、發佈的評論）也已經發生。

<strong>事前行動</strong>流程是在代理人執行風險步驟前插入閘口。代理人提出行動建議，閘口決定是否執行，只有在獲得批准時才會發生副作用。

| 方面 | 事後批准（README 程式碼片段） | 事前閘口（本筆記本） |
|---|---|---|
| 何時執行批准？ | 代理人產出結果後 | 任何副作用執行前 |
| 拒絕時的 LLM 費用 | 已支付 | 只支付提案費用，不支付行動費用 |
| 不可逆副作用 | 可能（行動已發生） | 已防止 |
| 稽核清晰度 | 批准是一個列印訊息 | 批准是帶時間戳、行動和原因的 JSON 紀錄 |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## 模式 2：風險分層

並非每個動作都需要人工批准。對公共 API 進行唯讀查詢和發送客戶電子郵件的風險不同。同等對待會浪費操作員的注意力並減慢代理速度。

一個簡單的三層模型：

| 階層 | 範例 | 批准流程 |
|---|---|---|
| `低`（唯讀） | 搜索知識庫、查詢航班選項、擷取公共網頁 | 自動執行，並記錄審計 |
| `中`（低成本變更） | 快取結果、增加計數器、安排提醒 | 自動執行，但每日批次審查 |
| `高`（對外或不可逆） | 發送電子郵件、扣款、發佈至公共頻道 | 阻止，需人工批准 |

這是一種分層。生產系統通常使用更細緻的分層（例如，AWS IAM 權限等級，基於角色的存取分層）。下述三層版本是用於混合唯讀和副作用動作的代理最小實用版本。

以下分類器使用關鍵字啟發式，讓示範保持確定性和低成本。在生產系統中，您會用訓練出的分類器或政策引擎替換此分類器。


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## 範例 3：審計日誌與修訂迴圈

一個 `print("Response approved.")` 不是審計日誌。為了信任，每個閘道決策都應該被記錄成結構化事件，讓你日後能查詢、重播或附加到事件審查中。

兩個部分：

1. **只能新增的 JSONL。** 每個決策一行，包含時間戳、行動、層級、決策及原因。方便 grep，也方便日後傳送到真正的日誌庫。
2. **拒絕時的修訂迴圈。** 當閘道回傳 `deny` 時，代理會用拒絕原因作為上下文重新提問自己，讓下一個提案能避免這個問題。


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 額外資源

還有其他多個公共項目實現了這些 HITL 模式的變體。比較不同方法以找出最適合您的堆疊方案：

- **LangChain** 人類在回路中的工具包裝器 ([docs](https://python.langchain.com/docs/integrations/tools/human_tools))：即插即用的工具包裝器，可暫停執行等待人類輸入。
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ 已重新架構)：使用一個代理角色專門代表多代理對話中的人類。
- **Microsoft Agent Framework (MAF)** 函式調用中介軟件 ([docs](https://learn.microsoft.com/agent-framework/))：在每個工具/函式調用周圍運行的中介軟件，適合用於門控邏輯和批准流程。

各個項目對這三個子模式的處理方式不同：LangChain 將它們包裝為工具，AutoGen 使用代理角色，而 Microsoft Agent Framework 則使用函式調用中介軟件。選擇自己的代理設計前，建議從頭到尾閱讀一兩個實現方案。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
